&nbsp;

&nbsp;

&nbsp;

&nbsp;

# GENCODE + FANTOM-CAT ID correspondence

- `gffcompare`で GENCODE v49 と FANTOM-CATの遺伝子IDを対応づける
- 例えば、今回、シスエレメントの関係性から、divergent っぽいよと議論している lncRNA が、実は FANTOM-CAT でも定義されているか可能性があるので、それをチェックする
- `./Tools/07_gffcompare.cwl`を中心に実行

In [1]:
import polars as pl
import datetime
import pathlib

from IPython.display import display

pl.Config.set_fmt_str_lengths(100)

polars.config.Config

In [2]:
VERSION = datetime.datetime.now().strftime("%y%m")
Execute_date = datetime.datetime.now().strftime("%y%m%d")
print(f"Execute Date: {Execute_date}")

Execute Date: 261010


&nbsp;

&nbsp;

&nbsp;

## 実行コマンド

- 自分で lv4 stringent の BEDファイルを liftover することにした


1. `cwltool --outdir ./Data/Database/FANTOM/FANTOM-CAT/ ./Tools/fantomcat/crossmap_bed.cwl ./config/fantomcat/crossmap_bed_lv4_stringent_hg19_to_hg38.yml`
2. `cwltool --outdir ./Data/Database/FANTOM/FANTOM-CAT/ ./Tools/fantomcat/prepare_bed12_for_bed2gtf.cwl ./config/fantomcat/prepare_bed12_lv4_stringent_hg38.yml`
3. `cwltool --outdir ./Data/Database/FANTOM/FANTOM-CAT/ ./Tools/fantomcat/bed2gtf.cwl ./config/fantomcat/bed2gtf_lv4_stringent_hg38.yml`

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## Check

- wallace さんから いただいた hg38 版の FANTOM-CAT と、自分で作成した lv4 stringent の FANTOM-CAT が正しいか確認する
- 実行コマンド
- `cwltool --outdir ./Data/Database/FANTOM/FANTOM-CAT/lv2_lv4_compare/ ./Tools/fantomcat/gffcompare.cwl ./config/fantomcat/gffcompare_lv4_stringent_vs_lv2_permissive_hg38.yml`

In [3]:
lv2_permissive_lv4_stringent_comparison = pl.read_csv(
    "../../Data/Database/FANTOM/FANTOM-CAT/lv2_lv4_compare/lv4_stringent_vs_lv2_permissive_hg38.tracking",
    separator="\t",
    has_header=False,
    new_columns=[
        "Entry",
        "Super_locus",
        "lv2_permissive",
        "class_code",
        "lv4_stringent"
    ]
)

display(lv2_permissive_lv4_stringent_comparison)

Entry,Super_locus,lv2_permissive,class_code,lv4_stringent
str,str,str,str,str
"""lv4_stringent_vs_lv2_permissive_hg38_00000001|1|5294""","""XLOC_000001""","""ENSG00000248527.1|FTMT20400000028.1""","""=""","""q1:ENSG00000248527.1|FTMT20400000028.1|1|0.000000|0.000000|0.000000|5294"""
"""lv4_stringent_vs_lv2_permissive_hg38_00000002|1|225""","""XLOC_000001""","""ENSG00000225972.1|ENST00000416931.1""","""=""","""q1:ENSG00000225972.1|ENST00000416931.1|1|0.000000|0.000000|0.000000|225"""
"""lv4_stringent_vs_lv2_permissive_hg38_00000003|2|438""","""XLOC_000001""","""ENSG00000225972.1|FTMT20300036855.1""","""=""","""q1:ENSG00000225972.1|FTMT20300036855.1|2|0.000000|0.000000|0.000000|438"""
"""lv4_stringent_vs_lv2_permissive_hg38_00000004|1|1236""","""XLOC_000001""","""ENSG00000237973.1|ENST00000414273.1""","""=""","""q1:ENSG00000237973.1|ENST00000414273.1|1|0.000000|0.000000|0.000000|1236"""
"""lv4_stringent_vs_lv2_permissive_hg38_00000005|1|625""","""XLOC_000001""","""ENSG00000229344.1|ENST00000427426.1""","""=""","""q1:ENSG00000229344.1|ENST00000427426.1|1|0.000000|0.000000|0.000000|625"""
…,…,…,…,…
"""lv4_stringent_vs_lv2_permissive_hg38_00260634|4|1385""","""XLOC_027373""","""CATG00000115107.1|FTMT29300001595.1""","""=""","""q1:CATG00000115107.1|FTMT29300001595.1|4|0.000000|0.000000|0.000000|1385"""
"""lv4_stringent_vs_lv2_permissive_hg38_00260635|3|651""","""XLOC_027373""","""CATG00000115107.1|HBMT00001592293.1""","""=""","""q1:CATG00000115107.1|HBMT00001592293.1|3|0.000000|0.000000|0.000000|651"""
"""lv4_stringent_vs_lv2_permissive_hg38_00260636|2|548""","""XLOC_027373""","""CATG00000115107.1|ENCT00000485163.1""","""=""","""q1:CATG00000115107.1|ENCT00000485163.1|2|0.000000|0.000000|0.000000|548"""


## Check

- tracking の 3 列目は `-r`（lv2 permissive）へのヒット、4 列目が class code、5 列目が query（lv4 stringent）です。いまの列名は逆なので、集計前に直します。
- 確認の見方:
- class code `=` が大半なら、lift 後のイントロン鎖は Wallace さんの hg38 と一致
- そのうち transcript_id も一致すれば、同じ転写産物として対応している
- `u` / `j` など `=` 以外は座標ずれ・alt contig・別アイソフォームの候補


In [4]:
# gffcompare tracking: col3 = reference hit (-r = lv2), col4 = class code, col5 = query (lv4)
parsed = lv2_permissive_lv4_stringent_comparison.with_columns(
    pl.col("lv2_permissive").str.split("|").list.get(1, null_on_oob=True).alias("lv2_transcript"),
    pl.col("lv4_stringent").str.split("|").list.get(1).alias("lv4_transcript"),
)

class_counts = (
    parsed.group_by("class_code")
    .agg(pl.len().alias("n"))
    .sort("n", descending=True)
)

display(class_counts)

class_code,n
str,u32
"""=""",260445
"""j""",60
"""u""",55
"""k""",37
"""c""",21
…,…
"""i""",5
"""p""",3
"""o""",2


In [5]:
equal = parsed.filter(pl.col("class_code") == "=")
equal_same_id = equal.filter(pl.col("lv2_transcript") == pl.col("lv4_transcript"))
print(
    f"query rows: {parsed.height}",
    f"class '=': {equal.height}",
    f"class '=' and same transcript_id: {equal_same_id.height}",
    f"not '=': {parsed.filter(pl.col('class_code') != '=').height}",
    sep="\n",
)

not_equal = parsed.filter(pl.col("class_code") != "=").select(
    "class_code",
    "lv2_permissive",
    "lv2_transcript",
    "lv4_transcript",
)

display(not_equal)

query rows: 260638
class '=': 260445
class '=' and same transcript_id: 258519
not '=': 193


class_code,lv2_permissive,lv2_transcript,lv4_transcript
str,str,str,str
"""c""","""ENSG00000134717.13|ENCT00000005800.1""","""ENCT00000005800.1""","""FTMT20400001926.1"""
"""c""","""CATG00000002414.1|ENCT00000006119.1""","""ENCT00000006119.1""","""FTMT20400002184.1"""
"""n""","""CATG00000015649.1|FTMT20300068901.1""","""FTMT20300068901.1""","""MICT00000018640.1"""
"""j""","""ENSG00000159208.11|ENST00000497211.1""","""ENST00000497211.1""","""ENST00000469255.1"""
"""k""","""ENSG00000197965.7|FTMT20300084159.1""","""FTMT20300084159.1""","""ENCT00000013704.1"""
…,…,…,…
"""u""","""-""",null,"""HBMT00000072051.1"""
"""c""","""ENSG00000223749.3|HBMT00001552355.1""","""HBMT00001552355.1""","""ENST00000385270.1"""
"""j""","""ENSG00000179542.11|ENST00000338017.4""","""ENST00000338017.4""","""ENST00000356928.1"""


&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## Load FANTOM-CAT lv4 stringent

- 実行コマンド
- `cwltool --outdir ./Data/Database/FANTOM/FANTOM-CAT/ ./Tools/06_pytximport_tx2gene.cwl ./config/tx2gene/pytximport_fantomcat_stringent_tx2gene.yml`

In [6]:
# fantomcat_stringent = pl.read_csv(
#     "../../Data/Database/FANTOM/FANTOM-CAT/fantomcat_lv4_stringent_hg19_tx2gene.tsv",
#     separator="\t"
# ).rename(
#     {
#         "transcript_id": "FANTOM-CAT_transcript",
#         "gene_id": "FANTOM-CAT_gene"
#     }
# )

# print(fantomcat_stringent.select("FANTOM-CAT_transcript").n_unique())
# display(fantomcat_stringent)

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## GENCODE release 49 vs. FANTOM-CAT lv4 stringent set

- 実行コマンド:
- `cwltool --outdir ./Data/Database/GENCODE_v49/GENCODE_FANTOMCAT_lv4_stringent_comparison/ ./Tools/fantomcat/gffcompare.cwl ./config/fantomcat/gffcompare_lv4_stringent_vs_gencode_v49_hg38.yml`

In [7]:
gffcompare_result = pl.read_csv(
    "../../Data/Database/GENCODE_v49/GENCODE_FANTOMCAT_lv4_stringent_comparison/lv4_stringent_vs_gencode_v49_hg38.tracking",
    separator="\t",
    has_header=False,
    new_columns=[
        "Entry",
        "Super_locus",
        "GENCODE",
        "Transcript_classification_codes",
        "FANTOM-CAT"
    ]
).with_columns([
    # No GENCODE hit is stored as "-"; split then has no transcript field
    pl.when(pl.col("GENCODE") != "-")
    .then(pl.col("GENCODE").str.split("|").list.get(0))
    .alias("GENCODE_gene"),
    pl.col("GENCODE").str.split("|").list.get(1, null_on_oob=True).alias("GENCODE_transcript"),
    pl.col("FANTOM-CAT").str.split("|").list.get(0).str.replace("q1:", "").alias("FANTOM-CAT_gene_lv4_stringent"),
    pl.col("FANTOM-CAT").str.split("|").list.get(1).alias("FANTOM-CAT_transcript_lv4_stringent"),
]).drop(
    "FANTOM-CAT",
    "GENCODE"
)

display(gffcompare_result)

Entry,Super_locus,Transcript_classification_codes,GENCODE_gene,GENCODE_transcript,FANTOM-CAT_gene_lv4_stringent,FANTOM-CAT_transcript_lv4_stringent
str,str,str,str,str,str,str
"""lv4_stringent_vs_gencode_v49_hg38_00000001|1|5294""","""XLOC_000001""","""k""","""ENSG00000237973.1""","""ENST00000414273.1""","""ENSG00000248527.1""","""FTMT20400000028.1"""
"""lv4_stringent_vs_gencode_v49_hg38_00000002|1|225""","""XLOC_000001""","""c""","""ENSG00000225972.1""","""ENST00000416931.1""","""ENSG00000225972.1""","""ENST00000416931.1"""
"""lv4_stringent_vs_gencode_v49_hg38_00000003|2|438""","""XLOC_000001""","""o""","""ENSG00000225972.1""","""ENST00000416931.1""","""ENSG00000225972.1""","""FTMT20300036855.1"""
"""lv4_stringent_vs_gencode_v49_hg38_00000004|1|1236""","""XLOC_000001""","""=""","""ENSG00000237973.1""","""ENST00000414273.1""","""ENSG00000237973.1""","""ENST00000414273.1"""
"""lv4_stringent_vs_gencode_v49_hg38_00000005|1|625""","""XLOC_000001""","""=""","""ENSG00000229344.1""","""ENST00000427426.1""","""ENSG00000229344.1""","""ENST00000427426.1"""
…,…,…,…,…,…,…
"""lv4_stringent_vs_gencode_v49_hg38_00260634|4|1385""","""XLOC_027955""","""j""","""ENSG00000229236.4""","""ENST00000755218.1""","""CATG00000115107.1""","""FTMT29300001595.1"""
"""lv4_stringent_vs_gencode_v49_hg38_00260635|3|651""","""XLOC_027955""","""j""","""ENSG00000229236.4""","""ENST00000755219.1""","""CATG00000115107.1""","""HBMT00001592293.1"""
"""lv4_stringent_vs_gencode_v49_hg38_00260636|2|548""","""XLOC_027955""","""c""","""ENSG00000229236.4""","""ENST00000666990.2""","""CATG00000115107.1""","""ENCT00000485163.1"""


&nbsp;

&nbsp;

&nbsp;

&nbsp;

## Stringent Filtering

- [gffcompare Transcript classification codes](https://ccb.jhu.edu/software/stringtie/gffcompare.shtml)
- `=` と `c` のみで作成する

In [8]:
gene_g = pl.col("GENCODE_gene")
tx_g = pl.col("GENCODE_transcript")
gene_f = pl.col("FANTOM-CAT_gene_lv4_stringent")
tx_f = pl.col("FANTOM-CAT_transcript_lv4_stringent")


# Strip trailing .version (e.g. ENSG00000123456.17 -> ENSG00000123456)
gene_g_id = gene_g.str.replace(r"\.\d+$", "")
tx_g_id = tx_g.str.replace(r"\.\d+$", "")
gene_f_id = gene_f.str.replace(r"\.\d+$", "")
tx_f_id = tx_f.str.replace(r"\.\d+$", "")


is_fantom_native = ~(
    tx_f.str.starts_with("ENST")
)

In [9]:
gffcompare_result_filter1 = gffcompare_result.filter(
    (pl.col("Transcript_classification_codes") == "=") |
    (pl.col("Transcript_classification_codes") == "c")
).with_columns(
    pl.when((gene_g == gene_f) & (tx_g == tx_f))
    .then(pl.lit("1_exact_version"))

    .when((gene_g_id == gene_f_id) & (tx_g_id == tx_f_id))
    .then(pl.lit("2_same_id_diff_version"))

    .when(gene_g_id == gene_f_id)
    .then(pl.lit("3_same_gene_diff_transcript"))

    .when(is_fantom_native, pl.col("Transcript_classification_codes") == "=")
    .then(pl.lit("4_equal_fantom_native"))

    .when(is_fantom_native, pl.col("Transcript_classification_codes") == "c")
    .then(pl.lit("5_c_fantom_native"))
    
    .otherwise(pl.lit("6_other"))

    .alias("id_correspondence_flag")
)

display(gffcompare_result_filter1)

Entry,Super_locus,Transcript_classification_codes,GENCODE_gene,GENCODE_transcript,FANTOM-CAT_gene_lv4_stringent,FANTOM-CAT_transcript_lv4_stringent,id_correspondence_flag
str,str,str,str,str,str,str,str
"""lv4_stringent_vs_gencode_v49_hg38_00000002|1|225""","""XLOC_000001""","""c""","""ENSG00000225972.1""","""ENST00000416931.1""","""ENSG00000225972.1""","""ENST00000416931.1""","""1_exact_version"""
"""lv4_stringent_vs_gencode_v49_hg38_00000004|1|1236""","""XLOC_000001""","""=""","""ENSG00000237973.1""","""ENST00000414273.1""","""ENSG00000237973.1""","""ENST00000414273.1""","""1_exact_version"""
"""lv4_stringent_vs_gencode_v49_hg38_00000005|1|625""","""XLOC_000001""","""=""","""ENSG00000229344.1""","""ENST00000427426.1""","""ENSG00000229344.1""","""ENST00000427426.1""","""1_exact_version"""
"""lv4_stringent_vs_gencode_v49_hg38_00000006|1|207""","""XLOC_000001""","""=""","""ENSG00000240409.1""","""ENST00000467115.1""","""ENSG00000240409.1""","""ENST00000467115.1""","""1_exact_version"""
"""lv4_stringent_vs_gencode_v49_hg38_00000007|1|842""","""XLOC_000001""","""=""","""ENSG00000248527.1""","""ENST00000514057.1""","""ENSG00000248527.1""","""ENST00000514057.1""","""1_exact_version"""
…,…,…,…,…,…,…,…
"""lv4_stringent_vs_gencode_v49_hg38_00260621|28|5547""","""XLOC_027954""","""=""","""ENSG00000012817.17""","""ENST00000541639.5""","""ENSG00000012817.11""","""ENST00000541639.1""","""2_same_id_diff_version"""
"""lv4_stringent_vs_gencode_v49_hg38_00260622|26|5099""","""XLOC_027954""","""=""","""ENSG00000012817.17""","""ENST00000382806.6""","""ENSG00000012817.11""","""ENST00000382806.2""","""2_same_id_diff_version"""
"""lv4_stringent_vs_gencode_v49_hg38_00260624|26|4685""","""XLOC_027954""","""=""","""ENSG00000012817.17""","""ENST00000440077.5""","""ENSG00000012817.11""","""ENST00000440077.1""","""2_same_id_diff_version"""


In [10]:
# check
equal_fantom_native = gffcompare_result_filter1.filter(pl.col("id_correspondence_flag") == "4_equal_fantom_native")


display(equal_fantom_native)

Entry,Super_locus,Transcript_classification_codes,GENCODE_gene,GENCODE_transcript,FANTOM-CAT_gene_lv4_stringent,FANTOM-CAT_transcript_lv4_stringent,id_correspondence_flag
str,str,str,str,str,str,str,str
"""lv4_stringent_vs_gencode_v49_hg38_00000098|3|2961""","""XLOC_000010""","""=""","""ENSG00000291156.2""","""ENST00000692821.3""","""ENSG00000217801.5""","""ENCT00000000183.1""","""4_equal_fantom_native"""
"""lv4_stringent_vs_gencode_v49_hg38_00000196|2|307""","""XLOC_000024""","""=""","""ENSG00000290916.2""","""ENST00000692936.2""","""ENSG00000160072.15""","""ENCT00000000337.1""","""4_equal_fantom_native"""
"""lv4_stringent_vs_gencode_v49_hg38_00000233|2|8378""","""XLOC_000029""","""=""","""ENSG00000293188.2""","""ENST00000417099.1""","""ENSG00000227775.3""","""MICT00000000998.1""","""4_equal_fantom_native"""
"""lv4_stringent_vs_gencode_v49_hg38_00000235|2|2807""","""XLOC_000029""","""=""","""ENSG00000293188.2""","""ENST00000577672.2""","""ENSG00000227775.3""","""MICT00000001000.1""","""4_equal_fantom_native"""
"""lv4_stringent_vs_gencode_v49_hg38_00000314|4|4032""","""XLOC_000037""","""=""","""ENSG00000287356.2""","""ENST00000659232.1""","""ENSG00000157933.9""","""MICT00000001154.1""","""4_equal_fantom_native"""
…,…,…,…,…,…,…,…
"""lv4_stringent_vs_gencode_v49_hg38_00260563|3|1311""","""XLOC_027946""","""=""","""ENSG00000286009.3""","""ENST00000662422.2""","""CATG00000115079.1""","""MICT00000383403.1""","""4_equal_fantom_native"""
"""lv4_stringent_vs_gencode_v49_hg38_00260564|3|1180""","""XLOC_027946""","""=""","""ENSG00000286009.3""","""ENST00000651143.2""","""CATG00000115079.1""","""MICT00000383404.1""","""4_equal_fantom_native"""
"""lv4_stringent_vs_gencode_v49_hg38_00260613|3|1236""","""XLOC_027952""","""=""","""ENSG00000291031.2""","""ENST00000761001.1""","""CATG00000115103.1""","""ENCT00000485100.1""","""4_equal_fantom_native"""


&nbsp;

&nbsp;

&nbsp;

&nbsp;

## FANTOM-CAT metadata

<!-- - Download from [FANTOM5 web page](https://fantom.gsc.riken.jp/5/suppl/Hon_et_al_2016/data/assembly/lv4_stringent/FANTOM_CAT.lv4_stringent.info_table.gene.tsv.gz) (2026/09/09 access) -->
- Download from [FANTOM-CAT Web page](https://fantom.gsc.riken.jp/cat/v1/#/genes) (2026/10/02 access)
- ここから取得した情報をアノテーション情報とする (gene level)

&nbsp;

In [11]:
# fantomcat_stringent_id_mapping = pl.read_csv(
#     "../../Data/Database/FANTOM/FANTOM-CAT/FANTOM_CAT.lv4_stringent.info_table.gene.tsv.gz",
#     separator="\t"
# ).rename(
#     {
#         "geneID": "FANTOM-CAT_gene_lv4_stringent"    
#     }
# )
# display(fantomcat_stringent_id_mapping)

In [12]:
fantomcat_gene_info = pl.read_csv(
    "../../Data/Database/FANTOM/FANTOM-CAT/metadata/fantom_cat_all_gene_info.tsv",
    separator="\t"
).rename(
    {
        "geneID": "FANTOM-CAT_gene_lv4_stringent"
    }
)

display(fantomcat_gene_info)

FANTOM-CAT_gene_lv4_stringent,gene name,alias symbol,gene class,DHS support,gene category,TIR conservation[RS score],exon conservation[RS score],trait vs cell enrichment[fdr],eQTL mRNA coexpression[fdr],dynamic expression[fdr]
str,str,str,str,str,str,f64,f64,f64,f64,f64
"""CATG00000005529.1""","""CATG00000005529.1""",null,"""lncRNA, divergent""","""enhancer""","""e-lncRNA""",null,200.7,0.00822,1.1062e-20,8.1600e-11
"""CATG00000011720.1""","""CATG00000011720.1""",null,"""lncRNA, intergenic""","""enhancer""","""e-lncRNA""",null,532.2,1.6500e-24,6.9187e-47,6.0100e-19
"""CATG00000029360.1""","""CATG00000029360.1""",null,"""lncRNA, intergenic""","""enhancer""","""e-lncRNA""",null,633.0,0.00336,2.4845e-31,0.0000118
"""CATG00000053087.1""","""CATG00000053087.1""",null,"""lncRNA, divergent""","""promoter""","""p-lncRNA, divergent""",null,230.1,0.0000026,1.0729e-52,1.0600e-11
"""CATG00000066599.1""","""CATG00000066599.1""",null,"""lncRNA, divergent""","""promoter""","""p-lncRNA, divergent""",null,383.3,5.7600e-46,1.7399e-47,2.9700e-11
…,…,…,…,…,…,…,…,…,…,…
"""ENSG00000272575.1""","""RP11-702B10.1""",null,"""lncRNA, intergenic""","""enhancer""","""e-lncRNA""",611.2,null,null,null,null
"""ENSG00000272660.1""","""RP11-145M9.6""",null,"""lncRNA, antisense""","""no""","""others""",349.8,null,null,null,null
"""ENSG00000272690.1""","""RP11-803B1.8""",null,"""lncRNA, divergent""","""promoter""","""p-lncRNA, divergent""",94.8,null,null,null,null


In [ ]:
gffcompare_result_filter1_join = gffcompare_result_filter1.join(
    fantomcat_gene_info,
    on="FANTOM-CAT_gene_lv4_stringent",
    how="inner"
).drop(
    "Entry",
    "Super_locus",
    "alias symbol"
)

# DO NOT REMOVE AND RUN THIS CELL!
# gffcompare_result_filter1_join.write_csv(
#     f"../../out/GENCODE_FANTOMCAT_comparison/GENCODE_v49_FANTOMCAT_lv4_stringent_comparison_v{Execute_date}.tsv",
#     separator="\t"
# )

pl.Config.set_tbl_rows(2)
display(gffcompare_result_filter1_join)

Transcript_classification_codes,GENCODE_gene,GENCODE_transcript,FANTOM-CAT_gene_lv4_stringent,FANTOM-CAT_transcript_lv4_stringent,id_correspondence_flag,gene name,gene class,DHS support,gene category,TIR conservation[RS score],exon conservation[RS score],trait vs cell enrichment[fdr],eQTL mRNA coexpression[fdr],dynamic expression[fdr]
str,str,str,str,str,str,str,str,str,str,f64,f64,f64,f64,f64
"""c""","""ENSG00000225972.1""","""ENST00000416931.1""","""ENSG00000225972.1""","""ENST00000416931.1""","""1_exact_version""","""MTND1P23""","""pseudogenes""","""promoter""","""others""",58.2,58.2,null,null,0.0261
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""c""","""ENSG00000229236.4""","""ENST00000666990.2""","""CATG00000115107.1""","""ENCT00000485163.1""","""5_c_fantom_native""","""CATG00000115107.1""","""uncertain coding""","""no""","""others""",null,null,null,null,0.00445


In [14]:
# check
equal_fantom_native_join = gffcompare_result_filter1_join.filter(
    (pl.col("id_correspondence_flag") == "4_equal_fantom_native") &
    (
        (pl.col("gene class") != "protein coding mRNAs") &
        (pl.col("gene class") != "pseudogenes")
    )
)

pl.Config.set_tbl_rows(60)
display(equal_fantom_native_join)

Transcript_classification_codes,GENCODE_gene,GENCODE_transcript,FANTOM-CAT_gene_lv4_stringent,FANTOM-CAT_transcript_lv4_stringent,id_correspondence_flag,gene name,gene class,DHS support,gene category,TIR conservation[RS score],exon conservation[RS score],trait vs cell enrichment[fdr],eQTL mRNA coexpression[fdr],dynamic expression[fdr]
str,str,str,str,str,str,str,str,str,str,f64,f64,f64,f64,f64
"""=""","""ENSG00000287396.2""","""ENST00000664694.2""","""CATG00000098374.1""","""HBMT00000001386.1""","""4_equal_fantom_native""","""CATG00000098374.1""","""uncertain coding""","""no""","""others""",null,null,null,null,7.4000e-166
"""=""","""ENSG00000287396.2""","""ENST00000776896.1""","""CATG00000098374.1""","""HBMT00000001387.1""","""4_equal_fantom_native""","""CATG00000098374.1""","""uncertain coding""","""no""","""others""",null,null,null,null,7.4000e-166
"""=""","""ENSG00000300565.1""","""ENST00000772751.1""","""CATG00000117923.1""","""FTMT20300000032.1""","""4_equal_fantom_native""","""CATG00000117923.1""","""lncRNA, intergenic""","""enhancer""","""e-lncRNA""",null,null,null,null,null
"""=""","""ENSG00000300565.1""","""ENST00000772752.1""","""CATG00000117923.1""","""FTMT20300001301.1""","""4_equal_fantom_native""","""CATG00000117923.1""","""lncRNA, intergenic""","""enhancer""","""e-lncRNA""",null,null,null,null,null
"""=""","""ENSG00000227634.4""","""ENST00000777821.1""","""ENSG00000238290.1""","""ENCT00000001033.1""","""4_equal_fantom_native""","""RP11-431K24.1""","""lncRNA, divergent""","""promoter""","""p-lncRNA, divergent""",251.1,null,null,null,0.0000193
"""=""","""ENSG00000227634.4""","""ENST00000670361.1""","""ENSG00000238290.1""","""MICT00000002578.1""","""4_equal_fantom_native""","""RP11-431K24.1""","""lncRNA, divergent""","""promoter""","""p-lncRNA, divergent""",251.1,null,null,null,0.0000193
"""=""","""ENSG00000230337.5""","""ENST00000715291.2""","""CATG00000016561.1""","""HBMT00000003436.1""","""4_equal_fantom_native""","""CATG00000016561.1""","""lncRNA, divergent""","""promoter""","""p-lncRNA, divergent""",575.2,546.9,null,1.6757e-69,0.0186
"""=""","""ENSG00000230337.5""","""ENST00000805051.1""","""CATG00000016561.1""","""MICT00000003054.1""","""4_equal_fantom_native""","""CATG00000016561.1""","""lncRNA, divergent""","""promoter""","""p-lncRNA, divergent""",575.2,546.9,null,1.6757e-69,0.0186
"""=""","""ENSG00000230337.5""","""ENST00000715289.1""","""CATG00000016561.1""","""MICT00000003055.1""","""4_equal_fantom_native""","""CATG00000016561.1""","""lncRNA, divergent""","""promoter""","""p-lncRNA, divergent""",575.2,546.9,null,1.6757e-69,0.0186
